In [3]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import time

# 1. Constants
FEATURE_COLS = ['unlock_count', 'convo_count', 'sms_sent']
ROLLING_WINDOW = 14
Z_CRISIS_THRESHOLD = 2.5

# 2. The Logic Fix (Optimized for Speed)
class StudentMonitor:
    def __init__(self):
        self.history = []     
        self.sensor_history = [] 

    def _sensor_anomaly_score(self, sensor_dict):
        if len(self.sensor_history) < 2: return 50.0
        
        # We use a simple list of dicts to avoid overhead
        past_values = self.sensor_history
        means = {f: np.mean([d[f] for d in past_values]) for f in FEATURE_COLS}
        stds = {f: np.std([d[f] for d in past_values]) or 1.0 for f in FEATURE_COLS}

        z_scores = [abs(sensor_dict.get(f, 0) - means[f]) / stds[f] for f in FEATURE_COLS]
        return min((np.mean(z_scores) / 2.0) * 100.0, 100.0)

    def process_new_day(self, day_label, sensor_dict):
        # Calculate anomaly BEFORE appending
        risk_score = self._sensor_anomaly_score(sensor_dict)
        self.sensor_history.append(sensor_dict)

        if len(self.history) >= 2:
            window = self.history[-ROLLING_WINDOW:]
            std = np.std(window, ddof=1) or 1.0
            z_score = (risk_score - np.mean(window)) / std
        else:
            z_score = 0.0

        status = "🔴 CRISIS" if z_score >= Z_CRISIS_THRESHOLD else "✅ STABLE"
        self.history.append(risk_score)
        
        return {"Day": day_label, "Risk": round(risk_score, 2), "Z": round(z_score, 2), "Status": status}

# 3. Running with Progress Bar
monitor = StudentMonitor()
data_points = [
    {"unlock_count": 10, "convo_count": 5, "sms_sent": 5},  # Day 1
    {"unlock_count": 11, "convo_count": 4, "sms_sent": 6},  # Day 2
    {"unlock_count": 80, "convo_count": 0, "sms_sent": 0}   # Day 3 (CRISIS SPIKE)
]

final_results = []
print("Starting Simulation...")

# The Progress Bar Loop
for i, data in enumerate(tqdm(data_points, desc="Processing Days")):
    res = monitor.process_new_day(f"Day {i+1}", data)
    final_results.append(res)
    time.sleep(0.1) # Small delay just to let the bar animate

# Display results
df = pd.DataFrame(final_results)
df

Starting Simulation...


Processing Days: 100%|██████████| 3/3 [00:00<00:00,  9.71it/s]


,Day,Risk,Z,Status
0,Day 1,50.0,0.0,✅ STABLE
1,Day 2,50.0,0.0,✅ STABLE
2,Day 3,100.0,50.0,🔴 CRISIS


In [4]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# 1. Full Repository Constants
FEATURE_COLS = [
    'unlock_count', 'avg_session_sec', 'total_unlocked_sec', 'night_unlocks',
    'total_dark_hrs', 'dark_fragments', 'longest_dark_streak_hrs', 'night_dark_hrs',
    'convo_count', 'total_convo_min', 'avg_convo_length_min',
    'incoming_calls', 'outgoing_calls', 'missed_calls', 'total_call_min',
    'sms_sent', 'sms_received',
    'charge_sessions', 'night_charge_hrs',
    'avg_nearby_devices'
]
ROLLING_WINDOW = 14
Z_CRISIS_THRESHOLD = 2.5

# 2. Production-Ready Monitor
class StudentMonitor:
    def __init__(self):
        self.history = []     
        self.sensor_history = [] 

    def _sensor_anomaly_score(self, sensor_dict):
        if len(self.sensor_history) < 2: 
            return 50.0 # Neutral start
        
        past_df = pd.DataFrame(self.sensor_history)
        means = past_df.mean()
        # Safety floor (1.0) prevents the math from breaking on Day 3
        stds = past_df.std().replace(0, 1.0).fillna(1.0)

        z_scores = []
        for feat in FEATURE_COLS:
            val = sensor_dict.get(feat, 0)
            z = abs(val - means.get(feat, 0)) / stds.get(feat, 1.0)
            z_scores.append(z)
            
        return min((np.mean(z_scores) / 2.0) * 100.0, 100.0)

    def process_new_day(self, day_label, sensor_dict):
        # Calculate anomaly BEFORE appending (Fixes Contamination)
        risk_score = self._sensor_anomaly_score(sensor_dict)
        self.sensor_history.append(sensor_dict)

        # Z-Score calculation with a window safety check
        if len(self.history) >= 2:
            window = self.history[-ROLLING_WINDOW:]
            std = np.std(window, ddof=1)
            if std < 1.0 or np.isnan(std): std = 1.0 # Safety floor
            z_score = (risk_score - np.mean(window)) / std
        else:
            z_score = 0.0

        status = "🔴 CRISIS" if z_score >= Z_CRISIS_THRESHOLD else "✅ STABLE"
        self.history.append(risk_score)
        
        return {"Day": day_label, "Risk": round(risk_score, 2), "Z": round(z_score, 2), "Status": status}

# 3. Generating Full Feature Data
monitor = StudentMonitor()
full_sim_data = []

# Day 1 & 2: Stable behavior
for i in range(2):
    day_data = {feat: 10.0 for feat in FEATURE_COLS}
    full_sim_data.append(day_data)

# Day 3: Crisis spike across ALL metrics
crisis_data = {feat: 60.0 for feat in FEATURE_COLS} 
full_sim_data.append(crisis_data)

# 4. Execution with Progress Bar
final_results = []
print(f"Simulating {len(FEATURE_COLS)} features...")

for i, data in enumerate(tqdm(full_sim_data, desc="Processing Full Audit")):
    res = monitor.process_new_day(f"Day {i+1}", data)
    final_results.append(res)

pd.DataFrame(final_results)

Simulating 20 features...


Processing Full Audit: 100%|██████████| 3/3 [00:00<00:00, 460.71it/s]


,Day,Risk,Z,Status
0,Day 1,50.0,0.0,✅ STABLE
1,Day 2,50.0,0.0,✅ STABLE
2,Day 3,100.0,50.0,🔴 CRISIS


Next Section: Realistic Math (Refining the Safety Floor)
Context: In the initial stress test, a Z-score of 50.0 was generated. While this correctly identified the crisis, it is a mathematical artifact of the "Perfectly Stable" Days 1 and 2.

In this section, we implement a Global Variance Floor (10.0). This represents "Normal Human Fluctuation" and ensures that Z-scores stay within a statistically sound range (e.g., 5.0) for the same 50-point risk jump.

In [5]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# 1. Update Monitor with 'Realistic' Variance Floor
class RealisticStudentMonitor(StudentMonitor): # Inheriting from your previous class
    def process_new_day(self, day_label, sensor_dict):
        # Calculate anomaly BEFORE appending (Fixes Contamination)
        risk_score = self._sensor_anomaly_score(sensor_dict)
        self.sensor_history.append(sensor_dict)

        if len(self.history) >= 2:
            window = self.history[-ROLLING_WINDOW:]
            actual_std = np.std(window, ddof=1)
            
            # --- THE REALISTIC FIX ---
            # We use 10.0 as a floor to represent 'Normal Daily Wiggle'.
            # This prevents Z-scores from exploding to 50.0.
            effective_std = max(actual_std, 10.0) 
            
            z_score = (risk_score - np.mean(window)) / effective_std
        else:
            z_score = 0.0

        status = "🔴 CRISIS" if z_score >= Z_CRISIS_THRESHOLD else "✅ STABLE"
        self.history.append(risk_score)
        
        return {"Day": day_label, "Risk": round(risk_score, 2), "Z": round(z_score, 2), "Status": status}

# 2. Run the Refined Test
refined_monitor = RealisticStudentMonitor()
data_points = [
    {feat: 10.0 for feat in FEATURE_COLS}, # Day 1
    {feat: 10.0 for feat in FEATURE_COLS}, # Day 2
    {feat: 60.0 for feat in FEATURE_COLS}  # Day 3 (Spike)
]

refined_results = []
for i, d in enumerate(tqdm(data_points, desc="Testing Realistic Math")):
    refined_results.append(refined_monitor.process_new_day(f"Day {i+1}", d))

pd.DataFrame(refined_results)

Testing Realistic Math: 100%|██████████| 3/3 [00:00<00:00, 1148.39it/s]


,Day,Risk,Z,Status
0,Day 1,50.0,0.0,✅ STABLE
1,Day 2,50.0,0.0,✅ STABLE
2,Day 3,100.0,5.0,🔴 CRISIS


🧪 Thorough Validation Test (The "Broken logic" Checker)
To be 100% sure this didn't break anything, we run this test that checks 3 things:

Purity: Does Day 1 affect Day 1's score? (It shouldn't).

Floor: Does a massive jump result in a reasonable Z-score (around 5.0)?

Recovery: If risk goes back to normal on Day 4, does the status return to STABLE?

In [9]:
import numpy as np
import pandas as pd

# Define these once at the top of your script/cell
FEATURE_COLS = [
    'unlock_count', 'avg_session_sec', 'total_unlocked_sec', 'night_unlocks',
    'total_dark_hrs', 'dark_fragments', 'longest_dark_streak_hrs', 'night_dark_hrs',
    'convo_count', 'total_convo_min', 'avg_convo_length_min',
    'incoming_calls', 'outgoing_calls', 'missed_calls', 'total_call_min',
    'sms_sent', 'sms_received',
    'charge_sessions', 'night_charge_hrs',
    'avg_nearby_devices'
]

class StudentMonitor:
    def __init__(self, uid, models=None):
        self.uid = uid
        self.models = models if models is not None else {}
        self.history = []          # Composite risk history
        self.sensor_history = []   # Raw sensor history
        
        # --- PERMANENT REPO CONFIGURATION ---
        self.ROLLING_WINDOW = 14
        self.Z_THRESHOLD_ELEVATED = 1.5
        self.Z_THRESHOLD_CRISIS = 2.5
        self.GLOBAL_VAR_FLOOR = 10.0 # Standard deviation proxy for "Cold Start"

    def _sensor_anomaly_score(self, sensor_dict):
        """Calculates deviation based on PAST history only."""
        if len(self.sensor_history) < 2: 
            return 50.0 # Neutral starting point
        
        past_df = pd.DataFrame(self.sensor_history)
        means = past_df.mean()
        
        # Apply the permanent floor to prevent division-by-zero on Day 3
        stds = past_df.std().replace(0, self.GLOBAL_VAR_FLOOR).fillna(self.GLOBAL_VAR_FLOOR)

        z_scores = []
        for feat in FEATURE_COLS:
            val = sensor_dict.get(feat, 0)
            z = abs(val - means.get(feat, 0)) / stds.get(feat, self.GLOBAL_VAR_FLOOR)
            z_scores.append(z)
            
        # Average Z-score across all 20 features
        return min((np.mean(z_scores) / 2.0) * 100.0, 100.0)

    def process_new_day(self, date_label, sensor_dict):
        # 1. ANOMALY: Calculate BEFORE appending (Fixes Contamination)
        sensor_anomaly = self._sensor_anomaly_score(sensor_dict)

        # 2. Z-SCORE: Proper Windowing with Realistic Variance Floor
        if len(self.history) >= 2:
            window = self.history[-self.ROLLING_WINDOW:]
            # Ensure the denominator reflects 'Normal Human Fluctuation'
            effective_std = max(np.std(window, ddof=1), self.GLOBAL_VAR_FLOOR)
            z_score = (sensor_anomaly - np.mean(window)) / effective_std
        else:
            z_score = 0.0

        # 3. DECISION: Tiered Logic using defined thresholds
        if z_score >= self.Z_THRESHOLD_CRISIS:
            status = "🔴 CRISIS"
        elif z_score >= self.Z_THRESHOLD_ELEVATED:
            status = "🟡 ELEVATED"
        else:
            status = "✅ STABLE"

        # 4. PERSISTENCE: Save AFTER calculation to preserve baseline purity
        self.sensor_history.append(sensor_dict.copy())
        self.history.append(sensor_anomaly)
        
        return {
            "Day": date_label, 
            "Risk": round(sensor_anomaly, 2), 
            "Z": round(z_score, 2), 
            "Status": status
        }

# --- THE COMPREHENSIVE AUDIT TEST ---
def run_final_validation():
    monitor = StudentMonitor(uid="final_audit_user")
    
    # Testing 4 Scenarios: Init -> Baseline -> Crisis Spike -> Recovery
    test_scenario = [
        {f: 10.0 for f in FEATURE_COLS}, # Day 1: Stable
        {f: 10.0 for f in FEATURE_COLS}, # Day 2: Stable
        {f: 60.0 for f in FEATURE_COLS}, # Day 3: CRISIS (Big Spike)
        {f: 10.0 for f in FEATURE_COLS}, # Day 4: RECOVERY (Back to 10)
    ]
    
    results = [monitor.process_new_day(f"Day {i+1}", d) for i, d in enumerate(test_scenario)]
    return pd.DataFrame(results)

# Execute the audit
run_final_validation()

,Day,Risk,Z,Status
0,Day 1,50.00,0.00,✅ STABLE
1,Day 2,50.00,0.00,✅ STABLE
2,Day 3,100.00,5.00,🔴 CRISIS
3,Day 4,28.87,-1.31,✅ STABLE
